# Notebook 09 — Agenti e il pattern ReAct

**Settimana:** 9
**Blocco:** 5.1 — Agenti autonomi
**Tempo stimato:** 4-5 ore

## Cosa costruirai

- Un loop di agente generico che funziona con qualsiasi insieme di strumenti
- Un agente di ricerca con 4 strumenti reali (Wikipedia, calcolatrice, data/ora, ricerca notizie simulata)
- Tracciabilità completa: ogni passo dell'agente viene registrato
- **Sfida finale:** agente di analisi finanziaria che risponde a domande sulle aziende usando più fonti

## Prerequisiti

Settimane 2 e 3 completate (function calling, gestione degli errori).

```bash
pip install openai anthropic requests python-dotenv
```

---
## Cella 0 — Setup

In [ ]:
import os
import json
import math
import uuid
import time
import datetime
import operator
import requests
from dataclasses import dataclass, field, asdict
from typing import Any, Callable
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

load_dotenv()

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

MODELO_OPENAI    = "gpt-4o-mini"
MODELO_ANTHROPIC = "claude-haiku-4-5-20251001"

print("Setup completato.")
print(f"OpenAI:    {MODELO_OPENAI}")
print(f"Anthropic: {MODELO_ANTHROPIC}")

---
## Cella 1 — Gli strumenti dell'agente

Definisci 4 strumenti reali.
Ogni strumento è una funzione Python + il suo schema JSON Schema per l'LLM.

In [ ]:
# ── Strumento 1: Wikipedia ────────────────────────────────────────────────────

def buscar_wikipedia(query: str, sentences: int = 3) -> str:
    """
    Cerca informazioni fattuali su Wikipedia.
    Restituisce un estratto dell'articolo più rilevante.
    """
    url    = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json", "list": "search",
        "srsearch": query, "srlimit": 1
    }
    try:
        resultados = requests.get(url, params=params, timeout=8).json()
        if not resultados["query"]["search"]:
            return f"Nessun risultato trovato per '{query}'."

        titulo  = resultados["query"]["search"][0]["title"]
        params2 = {
            "action": "query", "format": "json",
            "titles": titulo, "prop": "extracts",
            "exsentences": sentences, "exintro": True, "explaintext": True
        }
        pagina = requests.get(url, params=params2, timeout=8).json()
        pages  = pagina["query"]["pages"]
        texto  = list(pages.values())[0].get("extract", "Nessun contenuto.")
        return f"[Wikipedia: {titulo}]\n{texto}"
    except Exception as e:
        return f"Errore nella ricerca su Wikipedia: {e}"


# ── Strumento 2: Calcolatrice ─────────────────────────────────────────────────

def calcular(expresion: str) -> str:
    """
    Valuta un'espressione matematica in modo sicuro.
    Supporta +, -, *, /, **, sqrt, log, abs, round.
    """
    permitidos = {
        "__builtins__": {},
        "abs": abs, "round": round,
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "pi": math.pi, "e": math.e,
    }
    try:
        resultado = eval(expresion, permitidos)
        return f"{expresion} = {resultado}"
    except ZeroDivisionError:
        return "Errore: divisione per zero."
    except Exception as e:
        return f"Errore nel valutare '{expresion}': {e}"


# ── Strumento 3: Data e ora ───────────────────────────────────────────────────

def obtener_fecha_hora(zona_horaria: str = "UTC") -> str:
    """
    Restituisce la data e l'ora attuali in formato ISO.
    """
    ahora = datetime.datetime.utcnow()
    return json.dumps({
        "datetime_utc": ahora.isoformat(),
        "date":         ahora.strftime("%Y-%m-%d"),
        "time":         ahora.strftime("%H:%M:%S"),
        "weekday":      ahora.strftime("%A"),
        "timezone_note": f"Requested: {zona_horaria} (returned UTC)"
    })


# ── Strumento 4: Notizie simulate ─────────────────────────────────────────────
# In produzione questo chiamerebbe NewsAPI, Bing News o simili.
# Qui simuliamo con dati fissi per non aver bisogno di un'altra API key.

NOTICIAS_SIMULADAS = {
    "python": [
        "Python 3.13 released with JIT compiler improvements (Oct 2024)",
        "Python ranked #1 language in TIOBE index for 4th consecutive year",
    ],
    "ai": [
        "OpenAI releases GPT-4o with multimodal capabilities (May 2024)",
        "Anthropic raises $2.75B to develop Claude AI models",
        "EU AI Act enters into force, first comprehensive AI regulation",
    ],
    "fastapi": [
        "FastAPI 0.115 released with improved async support",
        "FastAPI surpasses 70k GitHub stars, most starred Python API framework",
    ],
    "default": [
        "No recent news found for this topic in the simulated database.",
    ]
}

def buscar_noticias(tema: str, max_resultados: int = 3) -> str:
    """
    Cerca notizie recenti su un tema.
    SIMULATO: in produzione si collega a NewsAPI o simili.
    """
    clave    = next((k for k in NOTICIAS_SIMULADAS if k in tema.lower()), "default")
    noticias = NOTICIAS_SIMULADAS[clave][:max_resultados]
    return f"[Notizie su '{tema}']\n" + "\n".join(f"- {n}" for n in noticias)


# ── Registro degli strumenti ──────────────────────────────────────────────────

FUNCIONES_DISPONIBLES: dict[str, Callable] = {
    "buscar_wikipedia": buscar_wikipedia,
    "calcular":         calcular,
    "obtener_fecha_hora": obtener_fecha_hora,
    "buscar_noticias":  buscar_noticias,
}

# Schemi JSON Schema per OpenAI
HERRAMIENTAS_OPENAI = [
    {
        "type": "function",
        "function": {
            "name": "buscar_wikipedia",
            "description": (
                "Search Wikipedia for factual background information, definitions, "
                "historical data, or explanations. Use for stable facts, NOT for "
                "real-time data like prices, weather, or breaking news."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query":     {"type": "string", "description": "Search term or question"},
                    "sentences": {"type": "integer", "description": "Number of sentences (1-10)", "default": 3}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular",
            "description": (
                "Evaluate a mathematical expression precisely. Supports +, -, *, /, **, "
                "sqrt(), log(), sin(), cos(), abs(), round(). Always use this instead "
                "of computing in your head."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {"type": "string", "description": "Valid Python math expression, e.g. 'sqrt(144) * 2.5'"}
                },
                "required": ["expresion"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obtener_fecha_hora",
            "description": "Get the current date and time. Use whenever the task requires knowing today's date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "zona_horaria": {"type": "string", "description": "Timezone name, e.g. 'Europe/Madrid'", "default": "UTC"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_noticias",
            "description": (
                "Search for recent news articles about a topic. Use for current events, "
                "product releases, company updates, or anything that changes frequently. "
                "Do NOT use for historical or encyclopedic information (use Wikipedia instead)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tema":           {"type": "string",  "description": "Topic to search for"},
                    "max_resultados": {"type": "integer", "description": "Max results (1-5)", "default": 3}
                },
                "required": ["tema"]
            }
        }
    }
]

# ── Test rapido degli strumenti ──────────────────────────────────────────────
print("Test degli strumenti:")
print(buscar_wikipedia("FastAPI", sentences=2))
print(calcular("sqrt(144) + 2**8"))
print(obtener_fecha_hora())
print(buscar_noticias("AI", max_resultados=2))

---
## Cella 2 — Il loop dell'agente (OpenAI)

Implementa il loop ReAct completo con tracciabilità.
L'agente chiama l'LLM, esegue gli strumenti richiesti e ripete finché non risponde direttamente.

In [ ]:
# ── Strutture di tracciabilità ────────────────────────────────────────────────

@dataclass
class PasoAgente:
    paso:        int
    tipo:        str          # "tool_call" | "parallel_tool_calls" | "final_answer" | "error"
    herramienta: str | None
    argumentos:  dict | None
    resultado:   str | None
    latencia_ms: float

@dataclass
class TrazaAgente:
    id:             str
    pregunta:       str
    respuesta:      str | None
    pasos:          list[PasoAgente] = field(default_factory=list)
    tokens_totales: int   = 0
    completado:     bool  = False
    razon_parada:   str   = ""

    def coste_usd(self) -> float:
        return self.tokens_totales * 0.00000015   # gpt-4o-mini circa

    def resumen(self) -> None:
        n_herramientas = sum(1 for p in self.pasos if p.tipo in ("tool_call", "parallel_tool_calls"))
        print(f"\n{'='*55}")
        print(f"ID:              {self.id}")
        print(f"Completato:      {self.completado} ({self.razon_parada})")
        print(f"Passi totali:    {len(self.pasos)} ({n_herramientas} chiamate a strumenti)")
        print(f"Token:           {self.tokens_totales}")
        print(f"Costo circa:     ${self.coste_usd():.6f}")
        print(f"{'='*55}")
        for p in self.pasos:
            if p.tipo in ("tool_call", "parallel_tool_calls"):
                args_str = json.dumps(p.argumentos, ensure_ascii=False)[:60]
                res_str  = str(p.resultado)[:60] if p.resultado else "—"
                print(f"  [{p.paso}] {p.herramienta}({args_str}...)")
                print(f"       → {res_str}...")
            else:
                print(f"  [{p.paso}] {p.tipo}")


# ── Esecutore di strumenti ────────────────────────────────────────────────────

def ejecutar_herramienta(nombre: str, argumentos: dict) -> str:
    """Chiama la funzione Python corrispondente e restituisce il risultato come stringa."""
    if nombre not in FUNCIONES_DISPONIBLES:
        return f"Errore: lo strumento '{nombre}' non esiste."
    try:
        return str(FUNCIONES_DISPONIBLES[nombre](**argumentos))
    except TypeError as e:
        return f"Errore negli argomenti di '{nombre}': {e}"
    except Exception as e:
        return f"Errore nell'esecuzione di '{nombre}': {e}"


# ── Loop dell'agente (OpenAI) ─────────────────────────────────────────────────

SYSTEM_AGENTE = """You are a research assistant with access to tools.
Always use tools to verify facts rather than relying on your training data.
When you have all the information needed, provide a clear, concise final answer.
Cite your sources (tool names and results) in the answer."""


def ejecutar_agente_openai(
    pregunta: str,
    herramientas: list = HERRAMIENTAS_OPENAI,
    max_pasos: int = 8,
    verbose: bool = True
) -> TrazaAgente:
    """
    Loop ReAct con OpenAI.
    L'LLM decide a ogni passo se usare gli strumenti o rispondere direttamente.
    """
    traza    = TrazaAgente(id=str(uuid.uuid4())[:8], pregunta=pregunta, respuesta=None)
    mensajes = [
        {"role": "system",  "content": SYSTEM_AGENTE},
        {"role": "user",    "content": pregunta}
    ]

    if verbose:
        print(f"\nAgente avviato: '{pregunta[:60]}...'" if len(pregunta) > 60 else f"\nAgente avviato: '{pregunta}'")

    for paso in range(1, max_pasos + 1):
        t_inicio = time.perf_counter()

        respuesta_llm = cliente_openai.chat.completions.create(
            model=MODELO_OPENAI,
            messages=mensajes,
            tools=herramientas,
            tool_choice="auto"
        )

        latencia_ms       = (time.perf_counter() - t_inicio) * 1000
        mensaje_asistente = respuesta_llm.choices[0].message
        traza.tokens_totales += respuesta_llm.usage.total_tokens

        # ── Senza strumenti: risposta finale ──────────────────────────────────
        if not mensaje_asistente.tool_calls:
            traza.pasos.append(PasoAgente(
                paso=paso, tipo="final_answer",
                herramienta=None, argumentos=None,
                resultado=mensaje_asistente.content, latencia_ms=round(latencia_ms, 1)
            ))
            traza.respuesta   = mensaje_asistente.content
            traza.completado  = True
            traza.razon_parada = "respuesta_directa"
            if verbose:
                print(f"  [Passo {paso}] Risposta finale generata.")
            break

        # ── Con strumenti: esegue ogni tool call ──────────────────────────────
        n_calls = len(mensaje_asistente.tool_calls)
        if verbose:
            herrs = ", ".join(tc.function.name for tc in mensaje_asistente.tool_calls)
            print(f"  [Passo {paso}] Strumenti richiesti ({n_calls}): {herrs}")

        # Aggiunge il messaggio dell'assistente alla cronologia
        mensajes.append(mensaje_asistente)

        # Esegue tutti gli strumenti richiesti
        for tool_call in mensaje_asistente.tool_calls:
            t_herramienta = time.perf_counter()
            nombre        = tool_call.function.name
            try:
                argumentos = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                argumentos = {}

            resultado    = ejecutar_herramienta(nombre, argumentos)
            lat_herr_ms  = (time.perf_counter() - t_herramienta) * 1000

            tipo_paso = "parallel_tool_calls" if n_calls > 1 else "tool_call"
            traza.pasos.append(PasoAgente(
                paso=paso, tipo=tipo_paso,
                herramienta=nombre, argumentos=argumentos,
                resultado=resultado[:200], latencia_ms=round(lat_herr_ms, 1)
            ))

            # Aggiunge il risultato alla cronologia come messaggio di tipo tool
            mensajes.append({
                "role":         "tool",
                "tool_call_id": tool_call.id,
                "content":      resultado
            })

            if verbose:
                print(f"    → {nombre}: {resultado[:80]}...")

    else:
        # Abbiamo esaurito max_pasos senza risposta diretta
        traza.razon_parada = "limite_pasos"
        traza.respuesta    = "L'agente ha superato il limite di passi senza arrivare a una risposta."

    return traza


# ── Test 1: compito semplice (uno strumento) ──────────────────────────────────
t1 = ejecutar_agente_openai("What is the capital of France and what is its population?")
t1.resumen()
print(f"\nRisposta: {t1.respuesta}")

---
## Cella 3 — Compiti che richiedono più strumenti

Testa l'agente con compiti che richiedono di combinare informazioni da diverse fonti.
Osserva se l'LLM usa parallel tool calls o chiamate sequenziali.

In [ ]:
PREGUNTAS_COMPLEJAS = [
    # Richiede Wikipedia + calcolatrice
    "The speed of light is approximately 299,792 km/s. How many seconds does it take "
    "light to travel from the Sun to Earth? (the Sun-Earth distance is about 149.6 million km)",

    # Richiede Wikipedia + notizie
    "What is FastAPI and what are the most recent updates about it?",

    # Richiede data + Wikipedia
    "How old is Python today? (it was first released on February 20, 1991)",
]

for i, pregunta in enumerate(PREGUNTAS_COMPLEJAS, 1):
    print(f"\n{'='*60}")
    print(f"COMPITO {i}")
    print(f"{'='*60}")
    traza = ejecutar_agente_openai(pregunta, verbose=True)
    traza.resumen()
    print(f"\nRisposta finale:\n{traza.respuesta}")

**Osservazioni** (modifica questa cella):

- In quali compiti ha usato parallel tool calls (più strumenti nello stesso passo)?
- C'è stato qualche compito in cui l'agente ha usato più passi del necessario?
- C'è qualche risposta che contiene informazioni che l'agente avrebbe dovuto verificare ma non ha verificato?

---
## Cella 4 — Lo stesso agente con Anthropic (Claude)

Implementa lo stesso loop ReAct per Anthropic.
La differenza chiave: i risultati degli strumenti vanno in messaggi `role: user`, non `role: tool`.

In [ ]:
# Converte gli schemi di OpenAI nel formato di Anthropic
def openai_tools_to_anthropic(herramientas_openai: list) -> list:
    """Adatta il formato degli strumenti di OpenAI a quello di Anthropic."""
    resultado = []
    for h in herramientas_openai:
        fn = h["function"]
        resultado.append({
            "name":         fn["name"],
            "description":  fn["description"],
            "input_schema": fn["parameters"]
        })
    return resultado

HERRAMIENTAS_ANTHROPIC = openai_tools_to_anthropic(HERRAMIENTAS_OPENAI)


def ejecutar_agente_anthropic(
    pregunta: str,
    herramientas: list = HERRAMIENTAS_ANTHROPIC,
    max_pasos: int = 8,
    verbose: bool = True
) -> TrazaAgente:
    """
    Loop ReAct con Anthropic Claude.
    Differenze rispetto a OpenAI:
    - stop_reason="tool_use" indica che l'LLM vuole usare gli strumenti
    - La cronologia usa role:user con tipo tool_result per i risultati
    """
    traza    = TrazaAgente(id=str(uuid.uuid4())[:8], pregunta=pregunta, respuesta=None)
    mensajes = [{"role": "user", "content": pregunta}]

    if verbose:
        print(f"\nAgente Claude: '{pregunta[:60]}...'" if len(pregunta) > 60 else f"\nAgente Claude: '{pregunta}'")

    for paso in range(1, max_pasos + 1):
        t_inicio = time.perf_counter()

        respuesta_llm = cliente_anthropic.messages.create(
            model=MODELO_ANTHROPIC,
            max_tokens=1024,
            system=SYSTEM_AGENTE,
            messages=mensajes,
            tools=herramientas
        )

        latencia_ms = (time.perf_counter() - t_inicio) * 1000
        uso         = respuesta_llm.usage
        traza.tokens_totales += uso.input_tokens + uso.output_tokens

        # ── Senza strumenti: risposta finale ──────────────────────────────────
        if respuesta_llm.stop_reason == "end_turn":
            texto_final = "".join(
                b.text for b in respuesta_llm.content if hasattr(b, "text")
            )
            traza.pasos.append(PasoAgente(
                paso=paso, tipo="final_answer",
                herramienta=None, argumentos=None,
                resultado=texto_final, latencia_ms=round(latencia_ms, 1)
            ))
            traza.respuesta    = texto_final
            traza.completado   = True
            traza.razon_parada = "respuesta_directa"
            if verbose:
                print(f"  [Passo {paso}] Risposta finale generata.")
            break

        # ── Con strumenti: stop_reason == "tool_use" ──────────────────────────
        tool_uses = [b for b in respuesta_llm.content if b.type == "tool_use"]
        n_calls   = len(tool_uses)

        if verbose:
            herrs = ", ".join(tu.name for tu in tool_uses)
            print(f"  [Passo {paso}] Strumenti ({n_calls}): {herrs}")

        # Aggiunge il messaggio dell'assistente alla cronologia
        mensajes.append({"role": "assistant", "content": respuesta_llm.content})

        # Esegue gli strumenti e costruisce la risposta come role:user
        resultados_herramientas = []
        for tool_use in tool_uses:
            t_herr       = time.perf_counter()
            nombre       = tool_use.name
            argumentos   = tool_use.input
            resultado    = ejecutar_herramienta(nombre, argumentos)
            lat_herr_ms  = (time.perf_counter() - t_herr) * 1000

            tipo_paso = "parallel_tool_calls" if n_calls > 1 else "tool_call"
            traza.pasos.append(PasoAgente(
                paso=paso, tipo=tipo_paso,
                herramienta=nombre, argumentos=argumentos,
                resultado=resultado[:200], latencia_ms=round(lat_herr_ms, 1)
            ))

            resultados_herramientas.append({
                "type":        "tool_result",
                "tool_use_id": tool_use.id,
                "content":     resultado
            })

            if verbose:
                print(f"    → {nombre}: {resultado[:80]}...")

        # In Anthropic, i risultati vanno in un messaggio role:user
        mensajes.append({"role": "user", "content": resultados_herramientas})

    else:
        traza.razon_parada = "limite_pasos"
        traza.respuesta    = "L'agente ha superato il limite di passi."

    return traza


# ── Confronta OpenAI vs Anthropic sullo stesso compito ────────────────────────
TAREA_COMPARATIVA = (
    "What is Python's current version, what are the most recent news about it, "
    "and how many years has Python existed? (it was first released in February 1991)"
)

print("\nOpenAI:")
traza_oai = ejecutar_agente_openai(TAREA_COMPARATIVA, verbose=True)

print("\nAnthropic:")
traza_ant = ejecutar_agente_anthropic(TAREA_COMPARATIVA, verbose=True)

# Tabella comparativa
print("\n" + "=" * 52)
print(f"{'METRICA':<25} {'OPENAI':>12} {'ANTHROPIC':>12}")
print("=" * 52)
for label, v_oai, v_ant in [
    ("Passi totali",  len(traza_oai.pasos), len(traza_ant.pasos)),
    ("Token totali",  traza_oai.tokens_totales, traza_ant.tokens_totales),
    ("Completato",    traza_oai.completado, traza_ant.completado),
]:
    print(f"{label:<25} {str(v_oai):>12} {str(v_ant):>12}")
print("=" * 52)

---
## Sfida finale — Agente di analisi tecnologica

Costruisci un agente specializzato che risponde a domande su una tecnologia
combinando informazioni enciclopediche, notizie recenti e calcoli.

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
# ── Strumento aggiuntivo: comparatore di tecnologie ──────────────────────────
# Questo strumento combina Wikipedia + struttura di confronto

def comparar_tecnologias(tecnologia_a: str, tecnologia_b: str, aspecto: str = "general") -> str:
    """
    Cerca informazioni su due tecnologie e restituisce i dati per confrontarle.
    """
    info_a = buscar_wikipedia(tecnologia_a, sentences=2)
    info_b = buscar_wikipedia(tecnologia_b, sentences=2)
    return json.dumps({
        "tecnologia_a": {"nombre": tecnologia_a, "info": info_a},
        "tecnologia_b": {"nombre": tecnologia_b, "info": info_b},
        "aspecto":      aspecto,
        "nota":         "Use this data to compare the two technologies based on the specified aspect."
    }, ensure_ascii=False)


HERRAMIENTAS_EXTENDIDAS_OPENAI = HERRAMIENTAS_OPENAI + [
    {
        "type": "function",
        "function": {
            "name": "comparar_tecnologias",
            "description": (
                "Look up and compare two technologies side by side on a specific aspect. "
                "Use this when asked to compare, contrast, or choose between two technologies."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tecnologia_a": {"type": "string", "description": "First technology name"},
                    "tecnologia_b": {"type": "string", "description": "Second technology name"},
                    "aspecto":      {"type": "string", "description": "Aspect to compare (performance, ecosystem, use cases, etc.)"}
                },
                "required": ["tecnologia_a", "tecnologia_b"]
            }
        }
    }
]

FUNCIONES_DISPONIBLES["comparar_tecnologias"] = comparar_tecnologias


# ── Agente specializzato (sfida) ──────────────────────────────────────────────

SYSTEM_TECH_ANALYST = """
You are a senior technology analyst specializing in software development tools and frameworks.
Your reports are:
- Fact-based: always verify with tools before stating facts
- Structured: use headers and bullet points
- Actionable: include a concrete recommendation
- Honest about uncertainty: if you can't verify something, say so

Always use the available tools to gather information. Do not rely on training data alone.
"""


def agente_analista(
    pregunta: str,
    max_pasos: int = 10,
    verbose: bool = True
) -> TrazaAgente:
    """
    Agente di analisi tecnologica.

    TODO: implementa questo agente usando ejecutar_agente_openai() come base,
    ma con:
    1. SYSTEM_TECH_ANALYST come prompt di sistema
    2. HERRAMIENTAS_EXTENDIDAS_OPENAI (include il nuovo strumento di confronto)
    3. Un pre-processing: prima di eseguire il loop, aggiungi al contesto
       la data attuale (chiamando obtener_fecha_hora()) in modo che l'agente
       sappia in quale momento sta lavorando

    Suggerimento: puoi riutilizzare ejecutar_agente_openai() direttamente se modifichi
    SYSTEM_AGENTE, oppure puoi duplicare il loop e personalizzarlo.
    """
    # TODO: implementa l'agente analista
    raise NotImplementedError("Implementa agente_analista()")


# ── Domande di test per l'agente analista ─────────────────────────────────────
PREGUNTAS_ANALISTA = [
    "Should I use FastAPI or Django for building a REST API? Give me a recommendation.",
    "What is the current state of Python in AI/ML? What are the most recent developments?",
    "PostgreSQL has been around for decades. How old is it exactly and is it still actively developed?",
]

# Decommenta quando hai implementato agente_analista()
# for pregunta in PREGUNTAS_ANALISTA:
#     print(f"\n{'='*65}")
#     traza = agente_analista(pregunta, verbose=True)
#     print(f"\nREPORT:\n{traza.respuesta}")
#     traza.resumen()

### Estensioni opzionali

1. **Agente con memoria tra sessioni:** salva la cronologia della conversazione in un file JSON e caricala all'avvio. Permette di fare domande di follow-up in una sessione successiva.

2. **Limite di costo dinamico:** aggiungi un parametro `max_coste_usd` all'agente. Prima di ogni chiamata all'LLM, calcola il costo accumulato e fermati se viene superato il limite.

3. **Strumento RAG:** integra la collezione `rag_docs` del notebook 07 come strumento dell'agente. L'agente potrà scegliere tra cercare su Wikipedia, cercare nelle notizie **o cercare nel tuo corpus privato**.

In [ ]:
# Estensione 3: scheletro per integrare RAG come strumento
# Decommenta e completa se hai la collezione rag_docs del notebook 07

# import chromadb
# chroma_cliente = chromadb.PersistentClient(path="./chroma_db")

# def buscar_documentacion_interna(query: str, k: int = 3) -> str:
#     """
#     Search the internal documentation corpus (FastAPI, REST, Pydantic, JWT, Docker, PostgreSQL).
#     Use this for technical documentation queries about these specific technologies.
#     """
#     try:
#         coleccion = chroma_cliente.get_collection("rag_docs")
#         # TODO: implementa il retrieval usando generar_embedding() del notebook 07
#         ...
#     except Exception as e:
#         return f"Internal docs not available: {e}"

# HERRAMIENTA_RAG = {
#     "type": "function",
#     "function": {
#         "name": "buscar_documentacion_interna",
#         "description": (
#             "Search the internal technical documentation corpus. "
#             "Topics covered: FastAPI, REST APIs, Pydantic, JWT, Docker, PostgreSQL. "
#             "Prefer this over Wikipedia for detailed technical queries on these topics."
#         ),
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "query": {"type": "string", "description": "Technical question or search term"},
#                 "k":     {"type": "integer", "description": "Number of results (1-5)", "default": 3}
#             },
#             "required": ["query"]
#         }
#     }
# }

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 09 — Settimana 9 del Piano di Formazione AI Engineering*